# PI-ERE Data Exploration

This notebook demonstrates the data ingestion and exploration pipeline for PI-ERE.

## Overview

We will:
1. Set up the environment
2. Ingest data from OSINT sources
3. Explore the data
4. Harmonize data into unified panel
5. Visualize time-series patterns

In [ ]:
# Imports
import sys
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from pi_ere.data.ingest import DataIngestionOrchestrator, get_default_regions
from pi_ere.data.sources import ACLEDSource, GDELTSource, WorldBankSource, CommoditiesSource
from pi_ere.data.harmonize import DataHarmonizer

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Configuration & Setup

In [ ]:
# Select regions to analyze
# Focus on a few key extractive regions for this demo
REGIONS = [
    'COD',  # Democratic Republic of Congo - copper, cobalt
    'MLI',  # Mali - gold
    'ZMB',  # Zambia - copper
    'NGA',  # Nigeria - oil
]

# Date range
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)

print(f"Regions: {', '.join(REGIONS)}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")

## 2. Data Ingestion

**Note:** This requires API keys for some sources. Make sure you have:
- ACLED_API_KEY (get from https://developer.acleddata.com)
- FRED_API_KEY (get from https://fred.stlouisfed.org/docs/api/api_key.html)

Set these in a `.env` file in the project root.

In [ ]:
# Initialize orchestrator
orchestrator = DataIngestionOrchestrator()

# Register data sources
# Comment out sources you don't have API keys for

# orchestrator.register_source(ACLEDSource())  # Requires ACLED_API_KEY
# orchestrator.register_source(GDELTSource())   # No key required
# orchestrator.register_source(WorldBankSource())  # No key required
orchestrator.register_source(CommoditiesSource())  # Requires FRED_API_KEY for full data

print("Registered sources:", list(orchestrator.sources.keys()))

In [ ]:
# Ingest data
# This may take a few minutes depending on date range and sources

data = orchestrator.ingest_all(
    start_date=START_DATE,
    end_date=END_DATE,
    regions=REGIONS,
    save_raw=True,
)

## 3. Explore Raw Data

In [ ]:
# Summary statistics
print("\nData Summary:")
print("=" * 80)

for source_name, df in data.items():
    print(f"\n{source_name.upper()}:")
    print(f"  Records: {len(df):,}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    if 'region' in df.columns:
        print(f"  Regions: {df['region'].nunique()}")

In [ ]:
# Example: Explore ACLED data (if available)
if 'acled' in data and not data['acled'].empty:
    acled_df = data['acled']
    
    print("ACLED Event Types:")
    print(acled_df['event_type'].value_counts())
    
    print("\nFatalities by Region:")
    print(acled_df.groupby('region')['fatalities'].sum().sort_values(ascending=False))

In [ ]:
# Example: Explore Commodities data
if 'commodities' in data and not data['commodities'].empty:
    comm_df = data['commodities']
    
    # Plot commodity prices
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()
    
    commodities = comm_df['commodity'].unique()[:4]  # First 4 commodities
    
    for i, commodity in enumerate(commodities):
        commodity_data = comm_df[comm_df['commodity'] == commodity]
        
        axes[i].plot(commodity_data['date'], commodity_data['price'])
        axes[i].set_title(commodity)
        axes[i].set_xlabel('Date')
        axes[i].set_ylabel('Price')
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 4. Data Harmonization

Combine data from different sources into a unified time-series panel.

In [ ]:
# Initialize harmonizer
harmonizer = DataHarmonizer(granularity='D')  # Daily granularity

# Harmonize data
harmonized = harmonizer.harmonize(
    data_sources=data,
    regions=REGIONS,
    start_date=START_DATE,
    end_date=END_DATE,
)

In [ ]:
# Inspect harmonized data
print("Harmonized Data:")
print(f"  Shape: {harmonized.shape}")
print(f"  Regions: {harmonized['region'].nunique()}")
print(f"  Features: {harmonized['feature_name'].nunique()}")
print(f"  Date range: {harmonized['date'].min()} to {harmonized['date'].max()}")

print("\nSample features:")
print(harmonized['feature_name'].unique()[:10])

In [ ]:
# Create engineered features
harmonized = harmonizer.create_features(harmonized)

## 5. Visualization

Visualize time-series patterns for selected regions and features.

In [ ]:
# Convert to wide format for visualization
wide_df = harmonized.pivot_table(
    index=['region', 'date'],
    columns='feature_name',
    values='value'
).reset_index()

In [ ]:
# Example: Plot ACLED event counts by region (if available)
if 'acled_event_count' in wide_df.columns:
    fig, ax = plt.subplots(figsize=(15, 6))
    
    for region in REGIONS:
        region_data = wide_df[wide_df['region'] == region]
        ax.plot(
            region_data['date'],
            region_data['acled_event_count'],
            label=region,
            alpha=0.7
        )
    
    ax.set_title('Conflict Events Over Time by Region', fontsize=14)
    ax.set_xlabel('Date')
    ax.set_ylabel('Event Count')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Save harmonized data
output_path = harmonizer.save_harmonized(harmonized, filename='demo_harmonized_data.parquet')
print(f"Saved harmonized data to: {output_path}")

## Next Steps

1. **Embedding Generation**: See `02_baseline_forecasting.ipynb` for time-series embeddings
2. **Baseline Models**: Compare ARIMA, Prophet against simple baselines
3. **Transformer Forecasting**: Implement Chronos/TimesFM for risk forecasting
4. **Anomaly Detection**: Build early warning system for operational disruptions
5. **Similarity Search**: Find historical analogues for current conditions